# Implementing Custom Acquisition Functions

[_Link to Tutorial on BoTorch_](https://botorch.org/docs/tutorials/custom_acquisition/)

_Drew Gjerstad_

**Includes content from BoTorch's documentation.**

In this tutorial, we review how to implement a custom acquisition function using BoTorch machinery. Specifically, we will be implementing the **Upper Confidence Bound (UCB)** acquisition function. The acquisition function balances the exploration-exploitation tradeoff by assigning a score of $\mu + \sqrt{\beta}*\sigma$ if the posterior distribution is normal with mean $\mu$ and variance $\sigma^2$. BoTorch provides the "analytic" version of this acquisition function within the `UpperConfidenceBound` class as well as a Monte carlo version in the `qUpperConfidenceBound` class, allowing for $q$-batches of size greater than 1. We will implement scalarized versions of both of these in the cells below.

## Setup
Before anything else, we load the necessary dependencies.

In [11]:
# Load Dependencies
import math
from typing import Optional

from botorch.acquisition.monte_carlo import MCAcquisitionFunction
from botorch.models.model import Model
from botorch.sampling.base import MCSampler
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils import t_batch_mode_transform
from torch import Tensor

## Implement Scalarized Version of q-UCB

First, we will implement a scalarized version of the q-UCB acquisition function. This is particularly useful if we are in a multi-output setting wherein we want to model the effect of a design on more than one metric. To do this, we will extend the q-UCB acquisition function to accept a multi-output model and then perform q-UCB on a scalarized version of the multiple outputs using a weight vector. Luckily, implementing a custom acquisition function is easy as one only needs to implement the constructor and a `forward` method.

In [2]:
# Implement Scalarized Version of q-UCB

class qScalarizedUpperConfidenceBound(MCAcquisitionFunction):
    """
    This class implements a scalarized version of q-UCB, allowing for
    multi-output models.
    """
    def __init__(
            self,
            model: Model,
            beta: Tensor,
            weights: Tensor,
            sampler: Optional[MCSampler] = None,
    ) -> None:
        super(MCAcquisitionFunction, self).__init__(model=model)
        if sampler is None:
            sampler = SobolQMCNormalSampler(sample_shape=torch.Size([512]))
            
        self.sampler = sampler
        self.register_buffer("beta", torch.as_tensor(beta))
        self.register_buffer("weights", torch.as_tensor(weights))

    @t_batch_mode_transform()
    def forward(self, X:Tensor) -> Tensor:
        """
        Evaluate the scalarized qUCB on the candidate set `X`.

        Args:
            X: A `(b) x q x d`-dim Tensor of `(b)` t-batches with `q` `d`-dim
               design points each.

        Returns:
            Tensor: A `(b)`-dim Tensor of Upper Confidence Bound values at the
                given design points `X`.
        """
        posterior = self.model.posterior(X)
        samples = self.get_posterior_samples(posterior)  # n x b x q x o
        scalarized_samples = samples.matmul(self.weights)  # n x b x q
        mean = posterior.mean  # b x q x o
        scalarized_mean = mean.matmul(self.weights)  # b x q
        ucb_samples = (
            scalarized_mean
            + math.sqrt(self.beta * math.pi / 2)
            * (scalarized_samples - scalarized_mean).abs()
        )
        return ucb_samples.max(dim=-1)[0].mean(dim=0)

Notice that the definition of `qScalarizedUpperConfidenceBound` is very close to the definition of `qUpperConfidenceBound` and only a few extra lines were needed to accommodate the scalarization of multiple outputs. The `@t_batch_mode_transform` decorator makes sure that `X` has an explicit t-batch dimension. We can achieve the same scalarization effect easily using `ScalarizedPosteriorTransform` (see the end of this tutorial).

## Test Your Custom Acquisition Function
Before attempting to connect this custom acquisition function to a Bayesian optimization loop, the best practice is to test it first. Thus, we will ensure that it properly evaluates on a compatible multi-output model using a basic multi-output `SingleTaskGP` model trained with synthetic data.

In [3]:
# Test Custom Acquisition Function
import torch

from botorch.fit import fit_gpytorch_mll
from botorch.models import SingleTaskGP
from botorch.utils import standardize
from gpytorch.mlls import ExactMarginalLogLikelihood

torch.set_default_dtype(torch.double)

# generate synthetic dataset
X = torch.rand(20, 2)
Y = torch.stack([torch.sin(X[:, 0]), torch.cos(X[:, 1])], -1)

# fit multi-output model
gp = SingleTaskGP(train_X=X, train_Y=Y)
mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
fit_gpytorch_mll(mll)

# initialize acquisition function
qSUCB = qScalarizedUpperConfidenceBound(gp, beta=0.1,
                                        weights=torch.tensor([0.1, 0.5]))

In [4]:
# evaluate on single q-batch with q=3
qSUCB(torch.rand(3, 2))

tensor([0.5381], grad_fn=<MeanBackward1>)

In [5]:
# batch-evaluate on two q-batches with q=3
qSUCB(torch.rand(2, 3, 2))

tensor([0.5608, 0.5553], grad_fn=<MeanBackward1>)

This acquisition function is able to handle multi-output models and is thus ready to be incorporated with an optimization loop. Since BoTorch makes it easy to swap out acquisition functions, we will not demonstrate that here. Instead, we will look at how to make a scalarized version of analytic UCB (with $q=1$).

## Implement Scalarized Version of Analytic UCB ($q=1$)
This version of analytic UCB will accommodate multi-output models, assuming a multivariate normal posterior and $q=1$. Notice that this new class, `ScalarizedUpperConfidenceBound` subclasses `AnalyticAcquisitionFunction` since we are not implementing a Monte Carlo batch version (as we did previously). Compared to the Monte Carlo version, rather than using weights on the samples, we will directly scalarize the mean vector $\boldsymbol{\mu}$ and covariance matrix $\boldsymbol{\Sigma}$. Then, we will apply standard UCB on the univariate normal distribution with mean $w^\intercal\boldsymbol{\mu}$ and variance $w^\intercal\boldsymbol{\Sigma}w$. Finally, we will also specify `expected_q=1` in the `@t_batch_mode_transform` decorator to ensure that `X` has `q=1`.

In [6]:
# Implement Scalarized Version of Analytic UCB
from botorch.acquisition import AnalyticAcquisitionFunction

class ScalarizedUpperConfidenceBound(AnalyticAcquisitionFunction):
    """
    This class implements a scalarized version of analytic UCB for use with
    multi-output models.
    """
    def __init__(
        self,
        model: Model,
        beta: Tensor,
        weights: Tensor,
        maximize: bool = True,
    ) -> None:
        super(AnalyticAcquisitionFunction, self).__init__(model)
        self.maximize = maximize
        self.register_buffer("beta", torch.as_tensor(beta))
        self.register_buffer("weights", torch.as_tensor(weights))

    @t_batch_mode_transform(expected_q=1)
    def forward(self, X:Tensor) -> Tensor:
        """
        Evaluate the Upper Confidence Bound on the candidate set `X` using
        scalarization.

        Args:
            X: A `(b) x d`-dim Tensor of `(b)` t-batches of `d`-dim design
                points each.
        
        Returns:
            A `(b)`-dim Tensor of Upper Confidence Bound values at the given
                design points `X`.
        """

        self.beta = self.beta.to(X)
        batch_shape = X.shape[:-2]
        posterior = self.model.posterior(X)
        means = posterior.mean.squeeze(dim=-2)  # b x o
        scalarized_mean = means.matmul(self.weights)  # b
        covs = posterior.mvn.covariance_matrix  # b x o x o
        weights = self.weights.view(
            1, -1, 1
        )  # 1 x o x 1 (assume single batch dimension)
        weights = weights.expand(batch_shape + weights.shape[1:])  # b x o x 1
        weights_transpose = weights.permute(0, 2, 1)  # b x 1 x o
        scalarized_variance = torch.bmm(
            weights_transpose, torch.bmm(covs, weights)
        ).view(
            batch_shape
        )  # b
        delta = (self.beta.expand_as(scalarized_mean) * scalarized_variance).sqrt()
        if self.maximize:
            return scalarized_mean + delta
        else:
            return scalarized_mean - delta

Once again, we will test our implementation of scalarized UCB and reuse the `SingleTaskGP` multi-output model trained on synthetic data from before. Note that we will pass in an explicit q-batch dimension for consistency, even though $q=1$.

In [7]:
# initialize acquisition function
SUCB = ScalarizedUpperConfidenceBound(gp, beta=0.1,
                                      weights=torch.tensor([0.1, 0.5]))

In [8]:
# evaluate on a single point
SUCB(torch.rand(1, 2))

tensor([0.5084], grad_fn=<AddBackward0>)

In [9]:
# batch-evaluate on 3 points
SUCB(torch.rand(3, 1, 2))

tensor([0.5408, 0.5562, 0.4936], grad_fn=<AddBackward0>)

## BoTorch Capability
As promised, we can use the `ScalarizedPosteriorTransform` class to easily and automatically handle the scalarization for us. This only requires a couple lines of code and this class can be used for both the analytic and Monte Carlo versions.

In [10]:
from botorch.acquisition.objective import ScalarizedPosteriorTransform
from botorch.acquisition.analytic import UpperConfidenceBound

pt = ScalarizedPosteriorTransform(weights=torch.tensor([0.1, 0.5]))
SUCB = UpperConfidenceBound(gp, beta=0.1, posterior_transform=pt)